In [1]:

import sys
import os

sys.path.append('..')

import pandas as pd
import scripts as kw

save_dir = 'test'
os.makedirs(save_dir, exist_ok=True)

train_df = pd.DataFrame([
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 100,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 101,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 102,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 103,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 104,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 105,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 106,
        kw.COLUMN_RATING: 4
    },
    {
        kw.COLUMN_USER_ID: 99,
        kw.COLUMN_ITEM_ID: 107,
        kw.COLUMN_RATING: 4
    },
    
    {
        kw.COLUMN_USER_ID: 100,
        kw.COLUMN_ITEM_ID: 100,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 100,
        kw.COLUMN_ITEM_ID: 101,
        kw.COLUMN_RATING: 0
    },
    
    {
        kw.COLUMN_USER_ID: 101,
        kw.COLUMN_ITEM_ID: 103,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 101,
        kw.COLUMN_ITEM_ID: 100,
        kw.COLUMN_RATING: 0
    },
    
    {
        kw.COLUMN_USER_ID: 102,
        kw.COLUMN_ITEM_ID: 100,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 102,
        kw.COLUMN_ITEM_ID: 101,
        kw.COLUMN_RATING: 5
    },
])

test_df = pd.DataFrame([
    {
        kw.COLUMN_USER_ID: 100,
        kw.COLUMN_ITEM_ID: 104,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 100,
        kw.COLUMN_ITEM_ID: 107,
        kw.COLUMN_RATING: 0
    },
    
    {
        kw.COLUMN_USER_ID: 101,
        kw.COLUMN_ITEM_ID: 104,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 101,
        kw.COLUMN_ITEM_ID: 107,
        kw.COLUMN_RATING: 0
    },
    
    {
        kw.COLUMN_USER_ID: 102,
        kw.COLUMN_ITEM_ID: 104,
        kw.COLUMN_RATING: 5
    },
    {
        kw.COLUMN_USER_ID: 102,
        kw.COLUMN_ITEM_ID: 107,
        kw.COLUMN_RATING: 5
    },
])

concat_df = pd.concat([train_df, test_df], ignore_index=True)

concat_df.to_csv(os.path.join(save_dir, 'test.csv'), sep=kw.DELIMITER, header=True, index=False, encoding=kw.ENCODING, quoting=kw.QUOTING, quotechar=kw.QUOTECHAR)

In [2]:
import numpy as np

target_embeddings = np.array([
    [1 ,  0],
    [0 ,  1],
    [-1,  0],
    [0 , -1],
    [1 ,  0],
    [0 ,  1],
    [-1,  0],
    [0 , -1],
])

context_embeddings = np.array([
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
    [0, 0],
])

np.save(os.path.join(save_dir, kw.FILE_ITEMS_EMBEDDINGS), target_embeddings)
np.save(os.path.join(save_dir, kw.FILE_CONTEXT_EMBEDDINGS), context_embeddings)

In [3]:
from scripts.modules.recommenders.Item2vec.Data_repr import DataRepr
import pickle

data_repr = DataRepr(train_df)

with open(os.path.join(save_dir, kw.FILE_SPARSE_REPR), 'wb') as f:
    pickle.dump(data_repr, f)

In [4]:
df = pd.read_csv(os.path.join(save_dir, 'test.csv'), delimiter=kw.DELIMITER, encoding=kw.ENCODING, quoting=kw.QUOTING, quotechar=kw.QUOTECHAR, header=0)
df = df.dropna().drop_duplicates(subset=[kw.COLUMN_USER_ID, kw.COLUMN_ITEM_ID], keep='last')

df

,id_user,id_item,rating
0,99,100,4
1,99,101,4
2,99,102,4
3,99,103,4
4,99,104,4
5,99,105,4
6,99,106,4
7,99,107,4
8,100,100,5
9,100,101,0


In [5]:
df_train = df.loc[:len(train_df)]

if kw.COLUMN_RATING in df_train.columns:
    explicit_ratings = df_train[kw.COLUMN_RATING]!=-1
    min_max = df_train[explicit_ratings][kw.COLUMN_RATING].agg(['min', 'max'])
    mean_rating = (min_max.loc['min'] + min_max.loc['max']) / 2  
    df_train = df_train[(df_train[kw.COLUMN_RATING]>=mean_rating)|(df_train[kw.COLUMN_RATING]==-1)]
df_train

,id_user,id_item,rating
0,99,100,4
1,99,101,4
2,99,102,4
3,99,103,4
4,99,104,4
5,99,105,4
6,99,106,4
7,99,107,4
8,100,100,5
10,101,103,5


In [6]:
df_test = df.loc[len(train_df):]
df_test

,id_user,id_item,rating
14,100,104,5
15,100,107,0
16,101,104,5
17,101,107,0
18,102,104,5
19,102,107,5
